# 07 Counterfactual Fairness, DiCE and Recourse Equity – Configurable Pipeline

## Purpose

This notebook generates counterfactual explanations using DiCE for unfavourable model predictions across German Credit, Pima Diabetes, and COMPAS.

Only the `ACTIVE_DATASET` value in the configuration block should be changed.

The notebook uses full pipeline models from `03_baseline_modeling`:

```text
original features → fitted preprocessor → trained classifier
```

Counterfactuals are generated in the original feature space because recourse should be interpretable in terms of real input variables.

## Link to EES

Outputs from this notebook feed into:

- `08_01_ees_framework_prepare_inputs`
- `08_04_ees_framework_actionability_score`
- `08_07_results_consolidation`

## Actionability Design

This version defines Actionability using three equal-weight components:

```text
Actionability Score =
(Availability Score + Proximity Score + Sparsity Score) / 3
```

### 1. Availability

Measures whether at least one valid counterfactual can be found.

```text
Availability Score =
number of unfavourable instances with at least one counterfactual
/
number of unfavourable instances
```

### 2. Proximity

Measures how close the nearest counterfactual is to the original instance using **numeric features only**.

Numeric distances are normalized using the training feature range:

```text
normalized distance =
absolute(original value - counterfactual value)
/
training feature range
```

Categorical features are intentionally excluded from proximity because distances such as `Rent → Own` or `Job A → Job B` require subjective assumptions that are difficult to defend consistently across domains.

The model-level proximity score is:

```text
Proximity Score = 1 - mean minimum normalized numeric distance
```

### 3. Sparsity

Measures how many features need to change.

```text
Sparsity Score =
1 - (mean number of changed features / total number of features)
```

Fewer changed features imply more understandable and practical recourse.

## Domain Interpretation

For German Credit and Pima Diabetes, counterfactuals may be interpreted as recourse or risk-reduction evidence, subject to domain caution.

For COMPAS, counterfactuals must be interpreted as **contestability and model sensitivity evidence only**, not as direct behavioural advice.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# Select only one active dataset.
# ACTIVE_DATASET = "german_credit"
ACTIVE_DATASET = "pima_diabetes"
# ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "preprocessing_folder": "02_data_preprocessing_german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_german_credit",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age", "Sex"],
        "counterfactual_interpretation": "Financial recourse evidence for unfavourable default predictions."
    },

    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "preprocessing_folder": "02_data_preprocessing_pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age"],
        "counterfactual_interpretation": "Clinical risk-reduction recourse evidence; not medical advice."
    },

    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "preprocessing_folder": "02_data_preprocessing_compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_compas",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["age", "sex", "race"],
        "counterfactual_interpretation": "Contestability and model sensitivity evidence; not behavioural advice."
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])
print("Unfavourable class:", CONFIG["unfavourable_class"], CONFIG["positive_class_label"])
print("Desired class:", CONFIG["desired_class"], CONFIG["negative_class_label"])
print("Immutable features:", CONFIG["immutable_features"])

Active dataset: Pima Diabetes
Dataset name: pima_diabetes
Unfavourable class: 1 Diabetes
Desired class: 0 No diabetes
Immutable features: ['Age']


In [2]:
# =========================================================
# 1. Install Required Packages
# =========================================================

try:
    import dice_ml
    DICE_AVAILABLE = True
except ImportError:
    print("Installing dice-ml...")
    !pip install -q dice-ml
    import dice_ml
    DICE_AVAILABLE = True

print("DiCE available:", DICE_AVAILABLE)

Installing dice-ml...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.7 MB/s eta 0:00:00
DiCE available: True


In [3]:
# =========================================================
# 2. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import time
import numpy as np
import pandas as pd
import joblib
import random

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [4]:
# =========================================================
# 3. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["counterfactual_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

PREPROCESSING_RESULTS_PATH = RESULTS_ROOT / CONFIG["preprocessing_folder"]
PREPROCESSING_DATASETS_PATH = PREPROCESSING_RESULTS_PATH / "datasets"

BASELINE_RESULTS_PATH = RESULTS_ROOT / CONFIG["baseline_folder"]
BASELINE_MODELS_PATH = BASELINE_RESULTS_PATH / "models"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Baseline models path:", BASELINE_MODELS_PATH)
print("Counterfactual output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes
Baseline models path: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/03_baseline_modeling_pima_diabetes/models
Counterfactual output path: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes


In [5]:
# =========================================================
# 4. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def load_target_series(path):
    target_df = read_csv_safely(path, "target file")

    if "target" in target_df.columns:
        return target_df["target"].astype(int)

    if CONFIG["target_name"] in target_df.columns:
        return target_df[CONFIG["target_name"]].astype(int)

    if target_df.shape[1] == 1:
        return target_df.iloc[:, 0].astype(int)

    raise ValueError(
        f"Could not identify target column in {path}. "
        f"Available columns: {target_df.columns.tolist()}"
    )


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def get_model_probabilities(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    preds = model.predict(X)
    return preds.astype(float)


def classify_actionability(score):
    if pd.isna(score):
        return "Not available"
    if score >= 0.80:
        return "High actionability"
    if score >= 0.60:
        return "Moderate-high actionability"
    if score >= 0.40:
        return "Moderate actionability"
    if score >= 0.20:
        return "Low-moderate actionability"
    return "Low actionability"


def calculate_changed_features(original_row, cf_row):
    changed_features = []

    for feature in original_row.index:
        if feature not in cf_row.index:
            continue

        original_value = original_row[feature]
        cf_value = cf_row[feature]

        if pd.isna(original_value) and pd.isna(cf_value):
            continue

        if str(original_value) != str(cf_value):
            changed_features.append(feature)

    return changed_features


def calculate_normalised_numeric_distance(original_row, cf_row, numeric_features, numeric_feature_ranges):
    distances = []

    for feature in numeric_features:
        if feature not in original_row.index or feature not in cf_row.index:
            continue

        original_value = original_row[feature]
        cf_value = cf_row[feature]

        if pd.isna(original_value) or pd.isna(cf_value):
            continue

        feature_range = numeric_feature_ranges.get(feature, 1.0)

        if feature_range == 0 or pd.isna(feature_range):
            feature_range = 1.0

        try:
            distance = abs(float(cf_value) - float(original_value)) / feature_range
            distances.append(distance)
        except Exception:
            continue

    if len(distances) == 0:
        return 0.0

    return float(np.mean(distances))


def clean_for_dice(df):
    cleaned = df.copy()

    for col in cleaned.columns:
        if cleaned[col].dtype == "object":
            cleaned[col] = cleaned[col].replace(["", " ", "nan", "NaN", "None"], np.nan)

    return cleaned

## 5. Load Original Modelling Data

Counterfactuals are generated in the original feature space.

Tracking columns are removed from the model feature matrix.

The notebook checks for missing values because DiCE cannot generate counterfactuals for query instances containing missing values.

In [6]:
# =========================================================
# 5. Load Original Modelling Data
# =========================================================

X_train_original = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_train_original.csv",
    "X_train_original"
)

X_test_original = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_test_original.csv",
    "X_test_original"
)

y_train = load_target_series(
    PREPROCESSING_DATASETS_PATH / "y_train.csv"
)

y_test = load_target_series(
    PREPROCESSING_DATASETS_PATH / "y_test.csv"
)

TRACKING_COLUMNS = [
    "original_index",
    "index",
    "Unnamed: 0"
]

X_train_model_original = (
    X_train_original
    .drop(columns=TRACKING_COLUMNS, errors="ignore")
    .copy()
)

X_test_model_original = (
    X_test_original
    .drop(columns=TRACKING_COLUMNS, errors="ignore")
    .copy()
)

X_train_model_original = clean_for_dice(X_train_model_original)
X_test_model_original = clean_for_dice(X_test_model_original)

print("X_train_original:", X_train_original.shape)
print("X_test_original:", X_test_original.shape)
print("X_train_model_original:", X_train_model_original.shape)
print("X_test_model_original:", X_test_model_original.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

display(X_train_model_original.head())

X_train_original: (614, 9)
X_test_original: (154, 9)
X_train_model_original: (614, 8)
X_test_model_original: (154, 8)
y_train: (614,)
y_test: (154,)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,1,90.000000,62.000000,12.000000,43.000000,27.200000,0.580000,24
1,5,126.000000,78.000000,27.000000,22.000000,29.600000,0.439000,40
2,2,105.000000,58.000000,40.000000,94.000000,34.900000,0.225000,25
3,1,146.000000,56.000000,NaN,NaN,29.700000,0.564000,29
4,0,95.000000,64.000000,39.000000,105.000000,44.600000,0.366000,22


In [7]:
# =========================================================
# 6. Identify Feature Types, Immutable Features and Ranges
# =========================================================

numeric_features = (
    X_train_model_original
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

categorical_features = [
    col for col in X_train_model_original.columns
    if col not in numeric_features
]

immutable_features = [
    feature for feature in CONFIG["immutable_features"]
    if feature in X_train_model_original.columns
]

mutable_features = [
    feature for feature in X_train_model_original.columns
    if feature not in immutable_features
]

numeric_feature_ranges = {}

for feature in numeric_features:
    feature_min = X_train_model_original[feature].min()
    feature_max = X_train_model_original[feature].max()
    feature_range = feature_max - feature_min

    if pd.isna(feature_range) or feature_range == 0:
        feature_range = 1.0

    numeric_feature_ranges[feature] = float(feature_range)

feature_type_df = pd.DataFrame([
    {
        "feature": feature,
        "data_type": "numeric" if feature in numeric_features else "categorical",
        "immutable": feature in immutable_features,
        "mutable": feature in mutable_features,
        "numeric_range_for_distance": numeric_feature_ranges.get(feature, np.nan)
    }
    for feature in X_train_model_original.columns
])

numeric_feature_range_df = pd.DataFrame([
    {
        "feature": feature,
        "numeric_range_for_distance": feature_range
    }
    for feature, feature_range in numeric_feature_ranges.items()
])

display(feature_type_df)
display(numeric_feature_range_df)

save_df(feature_type_df, "counterfactual_feature_type_register.csv")
save_df(numeric_feature_range_df, "numeric_feature_ranges_for_proximity.csv")

,feature,data_type,immutable,mutable,numeric_range_for_distance
0,Pregnancies,numeric,False,True,17.000000
1,Glucose,numeric,False,True,143.000000
2,BloodPressure,numeric,False,True,98.000000
3,SkinThickness,numeric,False,True,92.000000
4,Insulin,numeric,False,True,729.000000
5,BMI,numeric,False,True,48.900000
6,DiabetesPedigreeFunction,numeric,False,True,2.245000
7,Age,numeric,True,False,60.000000


,feature,numeric_range_for_distance
0,Pregnancies,17.000000
1,Glucose,143.000000
2,BloodPressure,98.000000
3,SkinThickness,92.000000
4,Insulin,729.000000
5,BMI,48.900000
6,DiabetesPedigreeFunction,2.245000
7,Age,60.000000


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_feature_type_register.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/numeric_feature_ranges_for_proximity.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/numeric_feature_ranges_for_proximity.csv')

## 7. Missing Value Audit

The model pipeline can handle missing values if the fitted preprocessor contains imputers.

However, DiCE query instances cannot contain missing values.

This notebook records rows with missing values and skips them during counterfactual generation, rather than silently failing.

If many rows are skipped, the recommended fix is to update Notebook `02_data_preprocessing` so that `X_train_original` and `X_test_original` are saved after the same imputation assumptions intended for DiCE.

In [8]:
# =========================================================
# 7. Missing Value Audit
# =========================================================

train_missing_summary_df = (
    X_train_model_original
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "n_missing_train"})
)

test_missing_summary_df = (
    X_test_model_original
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "n_missing_test"})
)

missing_feature_summary_df = train_missing_summary_df.merge(
    test_missing_summary_df,
    on="feature",
    how="outer"
)

missing_feature_summary_df["missing_in_train_or_test"] = (
    (missing_feature_summary_df["n_missing_train"] > 0) |
    (missing_feature_summary_df["n_missing_test"] > 0)
)

counterfactual_missing_input_rows_df = (
    X_test_model_original[
        X_test_model_original.isna().any(axis=1)
    ]
    .copy()
)

counterfactual_missing_input_rows_df["test_instance_position"] = counterfactual_missing_input_rows_df.index

display(missing_feature_summary_df)
display(counterfactual_missing_input_rows_df.head())

save_df(missing_feature_summary_df, "counterfactual_missing_feature_summary.csv")
save_df(counterfactual_missing_input_rows_df, "counterfactual_missing_input_rows.csv")

,feature,n_missing_train,n_missing_test,missing_in_train_or_test
0,Age,0,0,False
1,BMI,9,2,True
2,BloodPressure,23,12,True
3,DiabetesPedigreeFunction,0,0,False
4,Glucose,4,1,True
5,Insulin,290,84,True
6,Pregnancies,0,0,False
7,SkinThickness,175,52,True


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,test_instance_position
0,7,159.000000,64.000000,NaN,NaN,27.400000,0.294000,40,0
3,7,114.000000,64.000000,NaN,NaN,27.400000,0.732000,34,3
4,2,74.000000,NaN,NaN,NaN,NaN,0.102000,22,4
6,8,105.000000,100.000000,36.000000,NaN,43.300000,0.239000,45,6
7,8,197.000000,74.000000,NaN,NaN,25.900000,1.191000,39,7


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_missing_feature_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_missing_input_rows.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_missing_input_rows.csv')

## 8. Load Full Pipeline Models

DiCE uses full pipeline models saved by `03_baseline_modeling`.

Each pipeline accepts original feature values and internally applies the fitted preprocessor.

In [9]:
# =========================================================
# 8. Load Full Pipeline Models
# =========================================================

model_files = {
    "logistic_regression": "logistic_regression_pipeline.pkl",
    "decision_tree": "decision_tree_pipeline.pkl",
    "random_forest": "random_forest_pipeline.pkl",
    "xgboost": "xgboost_pipeline.pkl",
    "mlp": "mlp_pipeline.pkl",
}

pretty_model_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "mlp": "MLP",
}

pipeline_models = {}
model_load_rows = []

for model_id, filename in model_files.items():
    model_path = BASELINE_MODELS_PATH / filename

    if model_path.exists():
        pipeline_models[model_id] = joblib.load(model_path)
        found = True
        status = "Loaded"
    else:
        found = False
        status = "Skipped - pipeline file not found"

    model_load_rows.append({
        "model_id": model_id,
        "model_name": pretty_model_names[model_id],
        "model_file_found": found,
        "model_file_path": str(model_path),
        "status": status
    })

model_load_registry_df = pd.DataFrame(model_load_rows)

display(model_load_registry_df)

save_df(model_load_registry_df, "dice_model_load_registry.csv")

if not pipeline_models:
    raise ValueError("No full pipeline models were loaded.")

,model_id,model_name,model_file_found,model_file_path,status
0,logistic_regression,Logistic Regression,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
1,decision_tree,Decision Tree,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
2,random_forest,Random Forest,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
3,xgboost,XGBoost,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
4,mlp,MLP,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/dice_model_load_registry.csv


In [10]:
# =========================================================
# 8B. Create DiCE-Ready Original Datasets
# =========================================================

print("=" * 100)
print("Creating DiCE-ready original datasets")
print("=" * 100)

# ---------------------------------------------------------
# Use first loaded pipeline
# ---------------------------------------------------------

sample_pipeline = next(iter(pipeline_models.values()))

fitted_preprocessor = sample_pipeline.named_steps["preprocessor"]

# ---------------------------------------------------------
# Copy original data
# ---------------------------------------------------------

X_train_dice_original = X_train_model_original.copy()
X_test_dice_original = X_test_model_original.copy()

# ---------------------------------------------------------
# Recover imputation values from fitted preprocessor
# ---------------------------------------------------------

imputation_rows = []

for transformer_name, transformer, columns in fitted_preprocessor.transformers_:

    if transformer_name == "remainder":
        continue

    if hasattr(transformer, "named_steps"):

        if "imputer" in transformer.named_steps:

            fitted_imputer = transformer.named_steps["imputer"]

            for column, fill_value in zip(
                columns,
                fitted_imputer.statistics_
            ):

                X_train_dice_original[column] = (
                    X_train_dice_original[column]
                    .fillna(fill_value)
                )

                X_test_dice_original[column] = (
                    X_test_dice_original[column]
                    .fillna(fill_value)
                )

                imputation_rows.append({
                    "feature": column,
                    "imputation_value": str(fill_value),
                    "transformer": transformer_name
                })

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

dice_missing_summary_df = pd.DataFrame([
    {
        "dataset": "train",
        "missing_values_remaining":
        int(X_train_dice_original.isna().sum().sum())
    },
    {
        "dataset": "test",
        "missing_values_remaining":
        int(X_test_dice_original.isna().sum().sum())
    }
])

imputation_registry_df = pd.DataFrame(imputation_rows)

display(dice_missing_summary_df)
display(imputation_registry_df.head())

save_df(
    dice_missing_summary_df,
    "dice_missing_summary.csv"
)

save_df(
    imputation_registry_df,
    "dice_imputation_registry.csv"
)

print(
    "DiCE-ready datasets created successfully."
)

Creating DiCE-ready original datasets


,dataset,missing_values_remaining
0,train,0
1,test,0


,feature,imputation_value,transformer
0,Pregnancies,3.0,numeric
1,Glucose,117.0,numeric
2,BloodPressure,72.0,numeric
3,SkinThickness,29.0,numeric
4,Insulin,125.0,numeric


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/dice_missing_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/dice_imputation_registry.csv
DiCE-ready datasets created successfully.


## 9. Counterfactual Configuration

In [11]:
# =========================================================
# 9. Counterfactual Configuration
# =========================================================

TOTAL_CFS = 3
DICE_METHOD = "random"

counterfactual_config_df = pd.DataFrame([
    {
        "parameter": "RANDOM_STATE",
        "value": RANDOM_STATE,
        "description": "Fixed seed used for reproducible sampling where supported."
    },
    {
        "parameter": "TOTAL_CFS",
        "value": TOTAL_CFS,
        "description": "Number of counterfactuals requested per unfavourable instance."
    },
    {
        "parameter": "DICE_METHOD",
        "value": DICE_METHOD,
        "description": "DiCE generation method."
    },
    {
        "parameter": "unfavourable_class",
        "value": CONFIG["unfavourable_class"],
        "description": CONFIG["positive_class_label"]
    },
    {
        "parameter": "desired_class",
        "value": CONFIG["desired_class"],
        "description": CONFIG["negative_class_label"]
    },
    {
        "parameter": "immutable_features",
        "value": ", ".join(immutable_features),
        "description": "Features excluded from counterfactual changes."
    },
    {
        "parameter": "actionability_score_formula",
        "value": "(availability_score + proximity_score + sparsity_score) / 3",
        "description": "Equal-weight three-pillar actionability score."
    },
    {
        "parameter": "proximity_distance",
        "value": "minimum normalized numeric distance",
        "description": "Uses numeric features only; categorical changes are captured through sparsity."
    },
    {
        "parameter": "interpretation",
        "value": CONFIG["counterfactual_interpretation"],
        "description": "Domain-specific counterfactual interpretation."
    }
])

display(counterfactual_config_df)

save_df(counterfactual_config_df, "counterfactual_config.csv")

,parameter,value,description
0,RANDOM_STATE,42,Fixed seed used for reproducible sampling wher...
1,TOTAL_CFS,3,Number of counterfactuals requested per unfavo...
2,DICE_METHOD,random,DiCE generation method.
3,unfavourable_class,1,Diabetes
4,desired_class,0,No diabetes
5,immutable_features,Age,Features excluded from counterfactual changes.
6,actionability_score_formula,(availability_score + proximity_score + sparsi...,Equal-weight three-pillar actionability score.
7,proximity_distance,minimum normalized numeric distance,Uses numeric features only; categorical change...
8,interpretation,Clinical risk-reduction recourse evidence; not...,Domain-specific counterfactual interpretation.


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_config.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_config.csv')

## 10. Create DiCE Data Object

The DiCE data object combines the training features and target.

Continuous and categorical features are identified from the original feature space.

In [12]:
# =========================================================
# 10. Create DiCE Data Object
# =========================================================

train_dice_df = X_train_dice_original.copy()
train_dice_df["target"] = y_train.values

dice_data = dice_ml.Data(
    dataframe=train_dice_df,
    continuous_features=numeric_features,
    outcome_name="target"
)

dice_data_summary_df = pd.DataFrame([
    {
        "dataset_name": DATASET_NAME,
        "n_training_rows": train_dice_df.shape[0],
        "n_features": X_train_model_original.shape[1],
        "n_numeric_features": len(numeric_features),
        "n_categorical_features": len(categorical_features),
        "outcome_name": "target"
    }
])

display(dice_data_summary_df)

save_df(dice_data_summary_df, "dice_data_summary.csv")

,dataset_name,n_training_rows,n_features,n_numeric_features,n_categorical_features,outcome_name
0,pima_diabetes,614,8,8,0,target


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/dice_data_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/dice_data_summary.csv')

## 11. Generate Counterfactuals

For each model:

1. Predict test-set outcomes.
2. Select instances predicted as unfavourable.
3. Skip query rows with missing values because DiCE cannot process them.
4. Generate counterfactuals toward the desired class.
5. Calculate availability, minimum normalized numeric distance, and changed-feature count.

In [13]:
# =========================================================
# 11. Generate Counterfactuals
# =========================================================

prediction_rows = []
cf_rows = []
instance_summary_rows = []
error_rows = []
runtime_rows = []

for model_id, pipeline_model in pipeline_models.items():

    model_name = pretty_model_names[model_id]

    print("=" * 100)
    print(f"Generating counterfactuals for: {model_name}")
    print("=" * 100)

    start_time = time.time()

    try:
        # -------------------------------------------------
        # Predict using DiCE-ready original data
        # -------------------------------------------------
        y_pred = pipeline_model.predict(X_test_dice_original)
        y_proba = get_model_probabilities(
            pipeline_model,
            X_test_dice_original
        )

        for i in range(len(X_test_dice_original)):
            prediction_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "test_instance_position": i,
                "true_class": int(
                    y_test.iloc[i]
                    if hasattr(y_test, "iloc")
                    else y_test[i]
                ),
                "predicted_class": int(y_pred[i]),
                "predicted_probability": float(y_proba[i]),
                "is_unfavourable_prediction": (
                    int(y_pred[i]) == CONFIG["unfavourable_class"]
                ),
                "query_has_missing_values": bool(
                    X_test_dice_original.iloc[[i]]
                    .isna()
                    .sum()
                    .sum() > 0
                )
            })

        unfavourable_positions = [
            i for i, pred in enumerate(y_pred)
            if int(pred) == CONFIG["unfavourable_class"]
        ]

        dice_model = dice_ml.Model(
            model=pipeline_model,
            backend="sklearn"
        )

        dice_exp = dice_ml.Dice(
            dice_data,
            dice_model,
            method=DICE_METHOD
        )

        for test_pos in unfavourable_positions:

            query_instance = (
                X_test_dice_original
                .iloc[[test_pos]]
                .copy()
            )

            original_row = (
                X_test_dice_original
                .iloc[test_pos]
                .copy()
            )

            # -------------------------------------------------
            # Final safety check
            # -------------------------------------------------
            if query_instance.isna().sum().sum() > 0:

                error_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": test_pos,
                    "error_message": (
                        "Skipped - query instance contains missing values "
                        "after DiCE-ready imputation."
                    )
                })

                instance_summary_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": test_pos,
                    "counterfactual_found": False,
                    "skipped_due_to_missing_values": True,
                    "n_counterfactuals": 0,
                    "min_normalized_numeric_distance": np.nan,
                    "mean_normalized_numeric_distance": np.nan,
                    "min_changed_features": np.nan,
                    "mean_changed_features": np.nan
                })

                continue

            try:
                cf_result = dice_exp.generate_counterfactuals(
                    query_instance,
                    total_CFs=TOTAL_CFS,
                    desired_class=CONFIG["desired_class"],
                    features_to_vary=mutable_features,
                    random_seed=RANDOM_STATE
                )

                cf_df = (
                    cf_result
                    .cf_examples_list[0]
                    .final_cfs_df
                )

                if cf_df is None or cf_df.empty:

                    instance_summary_rows.append({
                        "model_id": model_id,
                        "model_name": model_name,
                        "test_instance_position": test_pos,
                        "counterfactual_found": False,
                        "skipped_due_to_missing_values": False,
                        "n_counterfactuals": 0,
                        "min_normalized_numeric_distance": np.nan,
                        "mean_normalized_numeric_distance": np.nan,
                        "min_changed_features": np.nan,
                        "mean_changed_features": np.nan
                    })

                    continue

                cf_distances = []
                cf_changed_counts = []

                for cf_rank, (_, cf_row) in enumerate(
                    cf_df.iterrows(),
                    start=1
                ):

                    if "target" in cf_row.index:
                        cf_feature_row = cf_row.drop(
                            labels=["target"]
                        )
                    else:
                        cf_feature_row = cf_row.copy()

                    changed_features = calculate_changed_features(
                        original_row=original_row,
                        cf_row=cf_feature_row
                    )

                    normalized_numeric_distance = (
                        calculate_normalised_numeric_distance(
                            original_row=original_row,
                            cf_row=cf_feature_row,
                            numeric_features=numeric_features,
                            numeric_feature_ranges=numeric_feature_ranges
                        )
                    )

                    cf_distances.append(
                        normalized_numeric_distance
                    )

                    cf_changed_counts.append(
                        len(changed_features)
                    )

                    cf_record = {
                        "model_id": model_id,
                        "model_name": model_name,
                        "test_instance_position": test_pos,
                        "cf_rank": cf_rank,
                        "normalized_numeric_distance": (
                            normalized_numeric_distance
                        ),
                        "n_changed_features": len(changed_features),
                        "changed_features": ", ".join(
                            changed_features
                        )
                    }

                    for feature in X_test_dice_original.columns:
                        cf_record[f"original_{feature}"] = (
                            original_row[feature]
                        )

                        cf_record[f"cf_{feature}"] = (
                            cf_feature_row[feature]
                            if feature in cf_feature_row.index
                            else np.nan
                        )

                    cf_rows.append(cf_record)

                instance_summary_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": test_pos,
                    "counterfactual_found": True,
                    "skipped_due_to_missing_values": False,
                    "n_counterfactuals": len(cf_df),
                    "min_normalized_numeric_distance": (
                        float(np.min(cf_distances))
                        if cf_distances
                        else np.nan
                    ),
                    "mean_normalized_numeric_distance": (
                        float(np.mean(cf_distances))
                        if cf_distances
                        else np.nan
                    ),
                    "min_changed_features": (
                        float(np.min(cf_changed_counts))
                        if cf_changed_counts
                        else np.nan
                    ),
                    "mean_changed_features": (
                        float(np.mean(cf_changed_counts))
                        if cf_changed_counts
                        else np.nan
                    )
                })

            except Exception as instance_error:

                error_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": test_pos,
                    "error_message": str(instance_error)
                })

                instance_summary_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": test_pos,
                    "counterfactual_found": False,
                    "skipped_due_to_missing_values": False,
                    "n_counterfactuals": 0,
                    "min_normalized_numeric_distance": np.nan,
                    "mean_normalized_numeric_distance": np.nan,
                    "min_changed_features": np.nan,
                    "mean_changed_features": np.nan
                })

        elapsed_seconds = time.time() - start_time

        runtime_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "elapsed_seconds": elapsed_seconds,
            "n_test_instances": len(X_test_dice_original),
            "n_unfavourable_predictions": len(
                unfavourable_positions
            )
        })

        print(
            f"Completed {model_name} in "
            f"{elapsed_seconds:.2f} seconds."
        )

        print(
            "Unfavourable predictions:",
            len(unfavourable_positions)
        )

    except Exception as model_error:

        error_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "test_instance_position": np.nan,
            "error_message": str(model_error)
        })

        print(
            f"Counterfactual generation failed for "
            f"{model_name}: {model_error}"
        )

counterfactual_prediction_summary_df = pd.DataFrame(
    prediction_rows
)

counterfactual_long_df = pd.DataFrame(
    cf_rows
)

counterfactual_instance_summary_df = pd.DataFrame(
    instance_summary_rows
)

counterfactual_error_log_df = pd.DataFrame(
    error_rows
)

counterfactual_runtime_diagnostics_df = pd.DataFrame(
    runtime_rows
)

display(counterfactual_prediction_summary_df.head())
display(counterfactual_instance_summary_df.head())
display(counterfactual_long_df.head())
display(counterfactual_error_log_df)

Generating counterfactuals for: Logistic Regression


100%|██████████| 1/1 [00:00<00:00,  9.63it/s]


Completed Logistic Regression in 5.47 seconds.
Unfavourable predictions: 46
Generating counterfactuals for: Decision Tree


100%|██████████| 1/1 [00:00<00:00,  9.75it/s]


Completed Decision Tree in 7.85 seconds.
Unfavourable predictions: 61
Generating counterfactuals for: Random Forest


...

Completed MLP in 5.22 seconds.
Unfavourable predictions: 44


,model_id,model_name,test_instance_position,true_class,predicted_class,predicted_probability,is_unfavourable_prediction,query_has_missing_values
0,logistic_regression,Logistic Regression,0,0,1,0.604280,True,False
1,logistic_regression,Logistic Regression,1,0,0,0.120747,False,False
2,logistic_regression,Logistic Regression,2,0,0,0.294740,False,False
3,logistic_regression,Logistic Regression,3,1,0,0.251213,False,False
4,logistic_regression,Logistic Regression,4,0,0,0.034765,False,False


,model_id,model_name,test_instance_position,counterfactual_found,skipped_due_to_missing_values,n_counterfactuals,min_normalized_numeric_distance,mean_normalized_numeric_distance,min_changed_features,mean_changed_features
0,logistic_regression,Logistic Regression,0,True,False,3,0.060237,0.070400,1.000000,1.333333
1,logistic_regression,Logistic Regression,7,True,False,3,0.068357,0.094512,1.000000,1.666667
2,logistic_regression,Logistic Regression,9,True,False,3,0.070041,0.114984,1.000000,1.666667
3,logistic_regression,Logistic Regression,16,True,False,3,0.023951,0.067256,1.000000,1.666667
4,logistic_regression,Logistic Regression,18,True,False,3,0.058654,0.065317,1.000000,1.333333


,model_id,model_name,test_instance_position,cf_rank,normalized_numeric_distance,n_changed_features,changed_features,original_Pregnancies,cf_Pregnancies,original_Glucose,cf_Glucose,original_BloodPressure,cf_BloodPressure,original_SkinThickness,cf_SkinThickness,original_Insulin,cf_Insulin,original_BMI,cf_BMI,original_DiabetesPedigreeFunction,cf_DiabetesPedigreeFunction,original_Age,cf_Age
0,logistic_regression,Logistic Regression,0,1,0.060237,2,"BloodPressure, BMI",7.000000,7.000000,159.000000,159.000000,64.000000,29.600000,29.000000,29.000000,125.000000,125.000000,27.400000,21.000000,0.294000,0.294000,40.000000,40.000000
1,logistic_regression,Logistic Regression,0,2,0.084353,1,Glucose,7.000000,7.000000,159.000000,62.500000,64.000000,64.000000,29.000000,29.000000,125.000000,125.000000,27.400000,27.400000,0.294000,0.294000,40.000000,40.000000
2,logistic_regression,Logistic Regression,0,3,0.066608,1,Glucose,7.000000,7.000000,159.000000,82.800000,64.000000,64.000000,29.000000,29.000000,125.000000,125.000000,27.400000,27.400000,0.294000,0.294000,40.000000,40.000000
3,logistic_regression,Logistic Regression,7,1,0.068357,1,Glucose,8.000000,8.000000,197.000000,118.800000,74.000000,74.000000,29.000000,29.000000,125.000000,125.000000,25.900000,25.900000,1.191000,1.191000,39.000000,39.000000
4,logistic_regression,Logistic Regression,7,2,0.110778,2,"Glucose, BloodPressure",8.000000,8.000000,197.000000,99.600000,74.000000,53.900000,29.000000,29.000000,125.000000,125.000000,25.900000,25.900000,1.191000,1.191000,39.000000,39.000000


""


## 12. Model-Level Three-Pillar Actionability Score

This section calculates the final Actionability Score using equal weights:

```text
Actionability Score =
(Availability Score + Proximity Score + Sparsity Score) / 3
```

- Higher availability is better.
- Lower normalized numeric distance is better.
- Fewer changed features are better.

In [14]:
# =========================================================
# 12. Model-Level Three-Pillar Actionability Score
# =========================================================

if counterfactual_prediction_summary_df.empty:
    unfavourable_prediction_summary_df = pd.DataFrame()
else:
    unfavourable_prediction_summary_df = (
        counterfactual_prediction_summary_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_test_instances=("test_instance_position", "nunique"),
            n_unfavourable_instances=("is_unfavourable_prediction", "sum"),
            mean_predicted_probability=("predicted_probability", "mean")
        )
    )

    missing_unfav_df = (
        counterfactual_prediction_summary_df[
            counterfactual_prediction_summary_df["is_unfavourable_prediction"] &
            counterfactual_prediction_summary_df["query_has_missing_values"]
        ]
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(n_unfavourable_with_missing_values=("test_instance_position", "nunique"))
    )

    unfavourable_prediction_summary_df = unfavourable_prediction_summary_df.merge(
        missing_unfav_df,
        on=["model_id", "model_name"],
        how="left"
    )

    unfavourable_prediction_summary_df["n_unfavourable_with_missing_values"] = (
        unfavourable_prediction_summary_df["n_unfavourable_with_missing_values"]
        .fillna(0)
        .astype(int)
    )

if counterfactual_instance_summary_df.empty:
    actionability_score_summary_df = pd.DataFrame()
else:
    actionability_score_summary_df = (
        counterfactual_instance_summary_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_unfavourable_instances=("test_instance_position", "nunique"),
            n_skipped_due_to_missing_values=("skipped_due_to_missing_values", "sum"),
            n_actionable_recourse_instances=("counterfactual_found", "sum"),
            mean_min_normalized_numeric_distance=("min_normalized_numeric_distance", "mean"),
            median_min_normalized_numeric_distance=("min_normalized_numeric_distance", "median"),
            mean_changed_features=("min_changed_features", "mean"),
            median_changed_features=("min_changed_features", "median")
        )
    )

    actionability_score_summary_df["availability_score"] = (
        actionability_score_summary_df["n_actionable_recourse_instances"] /
        actionability_score_summary_df["n_unfavourable_instances"]
    )

    actionability_score_summary_df["proximity_score"] = (
        1 - actionability_score_summary_df["mean_min_normalized_numeric_distance"]
    ).clip(0, 1)

    total_features = X_train_model_original.shape[1]

    actionability_score_summary_df["sparsity_score"] = (
        1 - (actionability_score_summary_df["mean_changed_features"] / total_features)
    ).clip(0, 1)

    actionability_score_summary_df["actionability_score"] = (
        actionability_score_summary_df["availability_score"] +
        actionability_score_summary_df["proximity_score"] +
        actionability_score_summary_df["sparsity_score"]
    ) / 3

    actionability_score_summary_df["actionability_score_pct"] = (
        actionability_score_summary_df["actionability_score"] * 100
    )

    actionability_score_summary_df["actionability_rank"] = (
        actionability_score_summary_df["actionability_score"]
        .rank(ascending=False, method="dense")
        .astype(int)
    )

    actionability_score_summary_df["actionability_band"] = (
        actionability_score_summary_df["actionability_score"]
        .apply(classify_actionability)
    )

    actionability_score_summary_df = (
        actionability_score_summary_df
        .sort_values(["actionability_rank", "model_id"])
        .reset_index(drop=True)
    )

display(unfavourable_prediction_summary_df)
display(actionability_score_summary_df)

,model_id,model_name,n_test_instances,n_unfavourable_instances,mean_predicted_probability,n_unfavourable_with_missing_values
0,decision_tree,Decision Tree,154,61,0.312580,0
1,logistic_regression,Logistic Regression,154,46,0.351548,0
2,mlp,MLP,154,44,0.331846,0
3,random_forest,Random Forest,154,44,0.341883,0
4,xgboost,XGBoost,154,51,0.336626,0


,model_id,model_name,n_unfavourable_instances,n_skipped_due_to_missing_values,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,median_min_normalized_numeric_distance,mean_changed_features,median_changed_features,availability_score,proximity_score,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band
0,decision_tree,Decision Tree,61,0,61,0.044152,0.039948,1.032787,1.000000,1.000000,0.955848,0.870902,0.942250,94.224985,1,High actionability
1,xgboost,XGBoost,51,0,51,0.047236,0.046642,1.098039,1.000000,1.000000,0.952764,0.862745,0.938503,93.850298,2,High actionability
2,random_forest,Random Forest,44,0,44,0.047352,0.044615,1.159091,1.000000,1.000000,0.952648,0.855114,0.935921,93.592064,3,High actionability
3,mlp,MLP,44,0,44,0.054787,0.053231,1.136364,1.000000,1.000000,0.945213,0.857955,0.934389,93.438902,4,High actionability
4,logistic_regression,Logistic Regression,46,0,46,0.056900,0.052710,1.217391,1.000000,1.000000,0.943100,0.847826,0.930309,93.030868,5,High actionability


## 13. Component Interpretation Table

In [15]:
# =========================================================
# 13. Component Interpretation Table
# =========================================================

actionability_component_interpretation_df = pd.DataFrame([
    {
        "component": "Availability Score",
        "question_answered": "Can recourse be found?",
        "calculation": "n_actionable_recourse_instances / n_unfavourable_instances",
        "higher_is_better": True
    },
    {
        "component": "Proximity Score",
        "question_answered": "How close is the nearest counterfactual?",
        "calculation": "1 - mean_min_normalized_numeric_distance",
        "higher_is_better": True
    },
    {
        "component": "Sparsity Score",
        "question_answered": "How few features need to change?",
        "calculation": "1 - (mean_changed_features / total_features)",
        "higher_is_better": True
    },
    {
        "component": "Final Actionability Score",
        "question_answered": "Is recourse available, close and simple?",
        "calculation": "(availability_score + proximity_score + sparsity_score) / 3",
        "higher_is_better": True
    }
])

display(actionability_component_interpretation_df)

save_df(actionability_component_interpretation_df, "actionability_component_interpretation.csv")

,component,question_answered,calculation,higher_is_better
0,Availability Score,Can recourse be found?,n_actionable_recourse_instances / n_unfavourab...,True
1,Proximity Score,How close is the nearest counterfactual?,1 - mean_min_normalized_numeric_distance,True
2,Sparsity Score,How few features need to change?,1 - (mean_changed_features / total_features),True
3,Final Actionability Score,"Is recourse available, close and simple?",(availability_score + proximity_score + sparsi...,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/actionability_component_interpretation.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/actionability_component_interpretation.csv')

## 14. Save Counterfactual Outputs

In [16]:
# =========================================================
# 14. Save Counterfactual Outputs
# =========================================================

outputs_to_save = [
    (counterfactual_prediction_summary_df, "counterfactual_prediction_summary.csv"),
    (counterfactual_instance_summary_df, "counterfactual_instance_summary.csv"),
    (counterfactual_long_df, "counterfactual_long_format.csv"),
    (unfavourable_prediction_summary_df, "unfavourable_prediction_summary.csv"),
    (actionability_score_summary_df, "actionability_score_summary.csv"),
    (actionability_score_summary_df, "ees_actionability_model_scores.csv"),
    (counterfactual_runtime_diagnostics_df, "counterfactual_runtime_diagnostics.csv"),
    (counterfactual_error_log_df, "counterfactual_error_log.csv"),
    (feature_type_df, "counterfactual_feature_type_register.csv"),
    (numeric_feature_range_df, "numeric_feature_ranges_for_proximity.csv"),
    (missing_feature_summary_df, "counterfactual_missing_feature_summary.csv"),
    (counterfactual_missing_input_rows_df, "counterfactual_missing_input_rows.csv"),
    (actionability_component_interpretation_df, "actionability_component_interpretation.csv")
]

for df, filename in outputs_to_save:
    save_df(df, filename)
    save_df(df, filename, folder=DATASETS_PATH)

save_excel(
    {
        "actionability_summary": actionability_score_summary_df,
        "component_interpretation": actionability_component_interpretation_df,
        "unfavourable_summary": unfavourable_prediction_summary_df,
        "prediction_summary": counterfactual_prediction_summary_df,
        "instance_summary": counterfactual_instance_summary_df,
        "counterfactual_long": counterfactual_long_df,
        "feature_type_register": feature_type_df,
        "numeric_ranges": numeric_feature_range_df,
        "missing_feature_summary": missing_feature_summary_df,
        "missing_input_rows": counterfactual_missing_input_rows_df,
        "config": counterfactual_config_df,
        "model_load_registry": model_load_registry_df,
        "runtime_diagnostics": counterfactual_runtime_diagnostics_df,
        "error_log": counterfactual_error_log_df
    },
    f"counterfactual_dice_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "unfavourable_class": CONFIG["unfavourable_class"],
    "desired_class": CONFIG["desired_class"],
    "positive_class_label": CONFIG["positive_class_label"],
    "negative_class_label": CONFIG["negative_class_label"],
    "total_counterfactuals_requested": TOTAL_CFS,
    "dice_method": DICE_METHOD,
    "immutable_features": immutable_features,
    "mutable_features": mutable_features,
    "actionability_formula": "(availability_score + proximity_score + sparsity_score) / 3",
    "proximity_definition": "1 - mean minimum normalized numeric distance; categorical features excluded from proximity.",
    "sparsity_definition": "1 - mean changed feature count divided by total features.",
    "interpretation": CONFIG["counterfactual_interpretation"],
    "standardisation_note": (
        "This counterfactual notebook uses the same configurable DiCE pipeline and three-pillar actionability "
        "score across German Credit, Pima Diabetes and COMPAS."
    )
}

with open(LOGS_PATH / "counterfactual_dice_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "counterfactual_dice_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All counterfactual outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_prediction_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/datasets/counterfactual_prediction_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_instance_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/datasets/counterfactual_instance_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables/counterfactual_long_format.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness

## 15. Interpretation Note

This notebook provides counterfactual recourse evidence for the EES Actionability pillar.

The revised Actionability Score is stronger than a simple recourse availability rate because it evaluates three distinct conditions:

- whether recourse exists,
- whether the nearest counterfactual is numerically close,
- whether the counterfactual requires few feature changes.

Categorical features are excluded from proximity because category-to-category distances are subjective, but they remain included in sparsity because categorical changes still increase the number of attributes that must change.

This provides a more defensible measure of whether model outcomes are practically contestable and actionable.

In [17]:
# =========================================================
# 15. Final Completion Message
# =========================================================

print("=" * 100)
print(f"07 Counterfactual DiCE completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- actionability_score_summary.csv")
print("- ees_actionability_model_scores.csv")
print("- counterfactual_prediction_summary.csv")
print("- counterfactual_instance_summary.csv")
print("- counterfactual_long_format.csv")
print("- unfavourable_prediction_summary.csv")
print("- numeric_feature_ranges_for_proximity.csv")
print(f"- counterfactual_dice_{DATASET_NAME}_outputs.xlsx")

07 Counterfactual DiCE completed successfully for: Pima Diabetes
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes/logs

Key outputs:
- actionability_score_summary.csv
- ees_actionability_model_scores.csv
- counterfactual_prediction_summary.csv
- counterfactual_instance_summary.csv
- counterfactual_long_format.csv
- unfavourable_prediction_summary.csv
- numeric_feature_ranges_for_proximity.csv
- counterfactual_dice_pima_diabetes_outputs.xlsx
